# ⬡ NeuralFind — Universal AI Recommendation Engine
> **Google Colab Notebook** | Python · Hugging Face · FAISS · Flask

This notebook builds a complete AI recommendation system for:
- 🎬 Movies · 🎮 Games · 📚 Courses · 📖 Books · 🛠️ Software

**Pipeline:**
1. Install packages
2. Create / load datasets
3. Generate semantic embeddings (sentence-transformers)
4. Build FAISS vector index
5. Test recommendation queries
6. Export files for Flask deployment

## 📦 Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
# sentence-transformers: semantic embedding models
# faiss-gpu: fast approximate nearest-neighbour search on GPU
# flask: web framework for deployment
# flask-sqlalchemy: ORM for SQLite database

!pip install -q sentence-transformers faiss-gpu flask flask-sqlalchemy pandas numpy scikit-learn
print('✅ Packages installed successfully')

## 🔧 Cell 2 — Imports & GPU Check

In [ ]:
import os, json, pickle, time
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('   ⚠️  No GPU found — will run on CPU (slower but works fine)')

## 📊 Cell 3 — Create Datasets

In [ ]:
# Create datasets directory
os.makedirs('datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)

# ── MOVIES ──────────────────────────────────────────────────
movies_data = [
    {'title':'Interstellar','category':'Movies','description':'A team of explorers travel through a wormhole in space. Epic sci-fi about time dilation, love, and human survival.','tags':'sci-fi space time-travel christopher-nolan','url':'https://www.imdb.com/title/tt0816692/'},
    {'title':'The Matrix','category':'Movies','description':'A computer hacker learns about the true nature of reality and his role in the war against its controllers. Cyberpunk action classic.','tags':'sci-fi cyberpunk action hacker virtual-reality','url':'https://www.imdb.com/title/tt0133093/'},
    {'title':'Arrival','category':'Movies','description':'A linguist works with the military to communicate with alien lifeforms. Explores language, time, and consciousness.','tags':'sci-fi aliens linguistics time','url':'https://www.imdb.com/title/tt2543164/'},
    {'title':'Blade Runner 2049','category':'Movies','description':'A young blade runner discovers a long-buried secret that could plunge society into chaos. Neo-noir sci-fi sequel.','tags':'sci-fi dystopian android future neo-noir','url':'https://www.imdb.com/title/tt1856101/'},
    {'title':'Dune','category':'Movies','description':'A noble family becomes embroiled in a war for control over the galaxy most valuable asset on a desert planet.','tags':'sci-fi epic fantasy desert adventure','url':'https://www.imdb.com/title/tt1160419/'},
    {'title':'Inception','category':'Movies','description':'A thief who steals corporate secrets through dream-sharing technology is given the inverse task of planting an idea.','tags':'sci-fi thriller dreams heist christopher-nolan','url':'https://www.imdb.com/title/tt1375666/'},
    {'title':'The Dark Knight','category':'Movies','description':'When the Joker wreaks havoc on Gotham, Batman must accept his greatest psychological and physical tests of heroism.','tags':'action superhero crime thriller','url':'https://www.imdb.com/title/tt0468569/'},
    {'title':'Oppenheimer','category':'Movies','description':'The story of J. Robert Oppenheimer and his role in the development of the atomic bomb during World War II.','tags':'drama history war biography christopher-nolan','url':'https://www.imdb.com/title/tt15398776/'},
    {'title':'Everything Everywhere All at Once','category':'Movies','description':'An aging Chinese immigrant is swept up in an insane adventure to save the world by exploring other universes.','tags':'sci-fi comedy drama multiverse','url':'https://www.imdb.com/title/tt6710474/'},
    {'title':'Parasite','category':'Movies','description':'Greed and class discrimination threaten the symbiotic relationship between the wealthy Park family and the destitute Kim clan.','tags':'drama thriller korean social-commentary','url':'https://www.imdb.com/title/tt6751668/'},
]

# ── GAMES ────────────────────────────────────────────────────
games_data = [
    {'title':'Elden Ring','category':'Games','description':'An open world action RPG set in the Lands Between. Explore a massive fantasy world with challenging combat and deep lore.','tags':'open-world RPG fantasy souls-like action','url':'https://store.steampowered.com/app/1245620/'},
    {'title':'The Elder Scrolls V Skyrim','category':'Games','description':'An open world action RPG where you play as the Dragonborn and battle dragons while exploring the vast province of Skyrim.','tags':'open-world RPG fantasy dragons adventure','url':'https://store.steampowered.com/app/489830/'},
    {'title':'The Witcher 3 Wild Hunt','category':'Games','description':'An open world RPG following Geralt of Rivia as he searches for his adopted daughter while battling monsters.','tags':'open-world RPG fantasy story-rich adventure','url':'https://store.steampowered.com/app/292030/'},
    {'title':'Cyberpunk 2077','category':'Games','description':'An open world action adventure story set in Night City, a megalopolis obsessed with power, glamour and body modification.','tags':'open-world RPG cyberpunk action sci-fi','url':'https://store.steampowered.com/app/1091500/'},
    {'title':'Baldurs Gate 3','category':'Games','description':'Gather your party and return to the Forgotten Realms in a tale of fellowship and betrayal, sacrifice and survival.','tags':'RPG fantasy turn-based DnD story-rich','url':'https://store.steampowered.com/app/1086940/'},
    {'title':'Hades','category':'Games','description':'Defy the god of the dead as you hack and slash out of the Underworld in this rogue-like dungeon crawler.','tags':'roguelike action dungeon-crawler indie mythology','url':'https://store.steampowered.com/app/1145360/'},
    {'title':'Minecraft','category':'Games','description':'A game about placing blocks and going on adventures. Explore randomly generated worlds and build amazing things.','tags':'sandbox survival building crafting creative','url':'https://www.minecraft.net/'},
    {'title':'Stardew Valley','category':'Games','description':'You have inherited your grandfather old farm plot. Build a farm, make friends, and explore caves in this indie RPG.','tags':'simulation farming RPG indie relaxing','url':'https://store.steampowered.com/app/413150/'},
    {'title':'Valorant','category':'Games','description':'A 5v5 character-based tactical shooter where precision gunplay meets unique agent abilities. Free to play competitive FPS.','tags':'FPS tactical shooter competitive multiplayer','url':'https://playvalorant.com/'},
    {'title':'Red Dead Redemption 2','category':'Games','description':'An epic tale of life in America unforgiving heartland. The game vast and atmospheric world will test your limits.','tags':'open-world adventure western story-rich','url':'https://store.steampowered.com/app/1174180/'},
]

# ── COURSES ──────────────────────────────────────────────────
courses_data = [
    {'title':'Machine Learning by Andrew Ng','category':'Courses','description':'The most popular machine learning course. Learn supervised learning, unsupervised learning and best practices from Stanford professor.','tags':'machine-learning AI python data-science beginner','url':'https://www.coursera.org/learn/machine-learning'},
    {'title':'Deep Learning Specialization','category':'Courses','description':'Master deep learning. Learn neural networks, improve deep learning skills and lead ML projects in this 5-course specialization.','tags':'deep-learning neural-networks AI TensorFlow python','url':'https://www.coursera.org/specializations/deep-learning'},
    {'title':'CS50 Introduction to Computer Science','category':'Courses','description':'Harvard introduction to computer science and the art of programming. Covers C, Python, SQL, web development and more.','tags':'programming computer-science beginner algorithms Harvard','url':'https://cs50.harvard.edu/'},
    {'title':'Python for Everybody','category':'Courses','description':'Learn to Program and Analyze Data with Python. Covers Python from basics to data access and manipulation techniques.','tags':'python programming beginner data-analysis','url':'https://www.coursera.org/specializations/python'},
    {'title':'Google Data Analytics Certificate','category':'Courses','description':'Gain data analytics skills using industry tools and platforms. Learn SQL, spreadsheets, R, and data visualization.','tags':'data-analytics SQL spreadsheets data-visualization Google','url':'https://www.coursera.org/professional-certificates/google-data-analytics'},
    {'title':'Complete Machine Learning and Data Science Bootcamp','category':'Courses','description':'Become a complete data scientist and machine learning engineer. Zero to mastery with Python, scikit-learn, pandas.','tags':'machine-learning data-science python sklearn pandas','url':'https://www.udemy.com/course/complete-machine-learning-and-data-science-zero-to-mastery/'},
    {'title':'Natural Language Processing Specialization','category':'Courses','description':'Master cutting-edge NLP techniques through four courses featuring lectures, labs and programming assignments.','tags':'NLP natural-language-processing AI deep-learning','url':'https://www.coursera.org/specializations/natural-language-processing'},
    {'title':'Ethical Hacking Course','category':'Courses','description':'Learn ethical hacking and penetration testing. Network scanning, exploitation, post-exploitation and reporting.','tags':'ethical-hacking penetration-testing cybersecurity security','url':'https://www.udemy.com/course/learn-ethical-hacking-from-scratch/'},
    {'title':'AWS Cloud Practitioner Essentials','category':'Courses','description':'Learn Amazon Web Services cloud fundamentals. Cloud concepts, AWS core services, security, architecture, pricing.','tags':'cloud-computing AWS Amazon beginner certification','url':'https://aws.amazon.com/training/'},
    {'title':'Introduction to Artificial Intelligence','category':'Courses','description':'Explore the basics of AI. Understand key concepts from machine learning, computer vision, to NLP and robotics.','tags':'AI artificial-intelligence beginner overview overview','url':'https://www.coursera.org/learn/introduction-to-ai'},
]

# ── BOOKS ─────────────────────────────────────────────────────
books_data = [
    {'title':'Hands-On Machine Learning with Scikit-Learn and TensorFlow','category':'Books','description':'A practical guide to machine learning using Python, Scikit-Learn, Keras and TensorFlow. Essential for ML practitioners.','tags':'machine-learning python tensorflow scikit-learn practical','url':'https://www.oreilly.com/library/view/hands-on-machine-learning/9781492032632/'},
    {'title':'Deep Learning by Ian Goodfellow','category':'Books','description':'The definitive textbook on deep learning. Covers mathematical foundations to advanced deep learning architectures.','tags':'deep-learning neural-networks AI mathematics','url':'https://www.deeplearningbook.org/'},
    {'title':'Clean Code by Robert Martin','category':'Books','description':'A handbook of agile software craftsmanship. Learn how to write clean, readable, maintainable code.','tags':'programming software-engineering clean-code best-practices','url':'https://www.amazon.com/Clean-Code-Handbook-Software-Craftsmanship/dp/0132350882'},
    {'title':'Artificial Intelligence A Modern Approach','category':'Books','description':'The most comprehensive introduction to AI. Covers search, planning, learning, knowledge representation and more.','tags':'AI artificial-intelligence textbook algorithms','url':'https://aima.cs.berkeley.edu/'},
    {'title':'Python Crash Course','category':'Books','description':'A hands-on project-based introduction to Python programming. Perfect for beginners who want to learn Python quickly.','tags':'python programming beginner projects','url':'https://nostarch.com/python-crash-course-3rd-edition'},
    {'title':'Cracking the Coding Interview','category':'Books','description':'189 programming interview questions and solutions. Prepare for software engineering interviews at top tech companies.','tags':'programming interviews algorithms data-structures career','url':'https://www.amazon.com/Cracking-Coding-Interview-Programming-Questions/dp/0984782850'},
    {'title':'Dune by Frank Herbert','category':'Books','description':'Set in the distant future. The story of Paul Atreides, whose family accepts the stewardship of the desert planet Arrakis.','tags':'sci-fi epic fantasy politics ecology classic','url':'https://www.amazon.com/Dune-Frank-Herbert/dp/0441013597'},
    {'title':'Atomic Habits','category':'Books','description':'An easy and proven way to build good habits and break bad ones. Tiny changes, remarkable results for productivity.','tags':'productivity habits self-improvement personal-development','url':'https://jamesclear.com/atomic-habits'},
    {'title':'Cybersecurity Essentials','category':'Books','description':'A foundational overview of cybersecurity concepts, tools and career paths. Perfect for those entering the field.','tags':'cybersecurity security fundamentals beginner','url':'https://www.amazon.com/Cybersecurity-Essentials-Charles-Brooks/dp/1119362393'},
    {'title':'Data Science for Business','category':'Books','description':'Written by renowned data science experts, helps business people understand key concepts in data science and ML.','tags':'data-science business analytics machine-learning','url':'https://www.amazon.com/Data-Science-Business-Data-Analytic-Thinking/dp/1449361323'},
]

# ── SOFTWARE ──────────────────────────────────────────────────
software_data = [
    {'title':'VS Code','category':'Software','description':'Visual Studio Code is a lightweight but powerful source code editor. Features debugging, Git control, extensions and more.','tags':'code-editor programming development IDE free','url':'https://code.visualstudio.com/'},
    {'title':'GitHub','category':'Software','description':'GitHub is where over 100 million developers shape the future of software. Host, version control and collaborate on code.','tags':'version-control git collaboration open-source devops','url':'https://github.com/'},
    {'title':'Docker','category':'Software','description':'Docker is a platform for developing, shipping and running applications in containers. Ensures consistency across environments.','tags':'containerization devops deployment infrastructure','url':'https://www.docker.com/'},
    {'title':'TensorFlow','category':'Software','description':'An end-to-end open source platform for machine learning. Comprehensive flexible ecosystem of tools, libraries and resources.','tags':'machine-learning deep-learning AI python open-source','url':'https://www.tensorflow.org/'},
    {'title':'PyTorch','category':'Software','description':'An open source machine learning framework that accelerates the path from research prototyping to production deployment.','tags':'machine-learning deep-learning AI python research','url':'https://pytorch.org/'},
    {'title':'Jupyter Notebook','category':'Software','description':'Open-source web application for creating documents with live code, equations, visualizations and narrative text.','tags':'data-science python machine-learning notebook','url':'https://jupyter.org/'},
    {'title':'Figma','category':'Software','description':'Figma is a collaborative web application for interface design. Create prototypes, wireframes and design systems.','tags':'design UI UX prototyping collaboration','url':'https://www.figma.com/'},
    {'title':'Tableau','category':'Software','description':'Tableau helps people transform data into actionable insights. Create beautiful and interactive data visualizations.','tags':'data-visualization analytics business-intelligence','url':'https://www.tableau.com/'},
    {'title':'Kali Linux','category':'Software','description':'An open-source Debian-based Linux distribution aimed at advanced penetration testing and security auditing.','tags':'cybersecurity penetration-testing ethical-hacking security linux','url':'https://www.kali.org/'},
    {'title':'Blender','category':'Software','description':'Free and open source 3D creation suite. Supports modeling, rigging, animation, simulation, rendering and compositing.','tags':'3D modeling animation visual-effects creative open-source','url':'https://www.blender.org/'},
]

# Save to CSV
for name, data in [('movies',movies_data),('games',games_data),('courses',courses_data),('books',books_data),('software',software_data)]:
    pd.DataFrame(data).to_csv(f'datasets/{name}.csv', index=False)

print('✅ All 5 datasets created!')
print(f'   Total items: {sum([len(d) for d in [movies_data,games_data,courses_data,books_data,software_data]])}')

## 🔗 Cell 4 — Merge & Clean All Datasets

In [ ]:
# Load and merge all CSV files into one master DataFrame
dfs = []
for name in ['movies','games','courses','books','software']:
    df = pd.read_csv(f'datasets/{name}.csv')
    df['source'] = name  # Track which file each record came from
    dfs.append(df)
    print(f'   {name}: {len(df)} records')

# Concatenate all dataframes
data = pd.concat(dfs, ignore_index=True)

# ── Data Cleaning ─────────────────────────────────────────────
# Fill any missing values with empty string
data.fillna('', inplace=True)

# Remove duplicate titles (keep first occurrence)
data.drop_duplicates(subset='title', keep='first', inplace=True)

# Strip whitespace from string columns
for col in ['title','category','description','tags']:
    data[col] = data[col].str.strip()

# ── Create Combined Text Field for Embedding ──────────────────
# This combined field is what gets embedded — includes title, category, description, and tags
# Weighted: title appears twice (more important for matching)
data['combined_text'] = (
    data['title'] + ' ' + data['title'] + ' ' +   # title repeated for higher weight
    data['category'] + ' ' +
    data['description'] + ' ' +
    data['tags']
)

print(f'\n✅ Merged dataset: {len(data)} total records')
print('\nCategory distribution:')
print(data['category'].value_counts().to_string())
data.head(3)

## 🤖 Cell 5 — Load Hugging Face Embedding Model

In [ ]:
# We use sentence-transformers/all-MiniLM-L6-v2:
# - Only 22M parameters — very lightweight and fast
# - 384-dimensional dense embeddings
# - Optimized for semantic similarity tasks
# - Excellent performance for recommendation retrieval

MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
# Alternatives (uncomment to use):
# MODEL_NAME = 'BAAI/bge-small-en-v1.5'       # 33M params, slightly better quality
# MODEL_NAME = 'intfloat/e5-small-v2'          # 33M params, strong for retrieval

print(f'📥 Loading model: {MODEL_NAME}')
start = time.time()

model = SentenceTransformer(MODEL_NAME, device=device)

print(f'✅ Model loaded in {time.time()-start:.1f}s')
print(f'   Embedding dimension: {model.get_sentence_embedding_dimension()}')
print(f'   Max sequence length: {model.max_seq_length}')
print(f'   Running on: {device.upper()}')

## ⚡ Cell 6 — Generate Embeddings (Batch Processing)

In [ ]:
# Generate embeddings for all records using batch processing
# Batching is crucial for GPU efficiency — avoids memory overflow
# batch_size=64 is optimal for Colab T4 GPU

BATCH_SIZE = 64 if device == 'cuda' else 32

print(f'🔄 Generating embeddings for {len(data)} records...')
print(f'   Batch size: {BATCH_SIZE}')
print(f'   Expected shape: ({len(data)}, {model.get_sentence_embedding_dimension()})')

start = time.time()

embeddings = model.encode(
    data['combined_text'].tolist(),
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2 normalize for cosine similarity via dot product
)

# Convert to float32 (FAISS requirement)
embeddings = embeddings.astype(np.float32)

print(f'\n✅ Embeddings generated in {time.time()-start:.1f}s')
print(f'   Shape: {embeddings.shape}')
print(f'   Memory: {embeddings.nbytes / 1024:.1f} KB')
print(f'   Dtype: {embeddings.dtype}')

## 🗄️ Cell 7 — Build FAISS Vector Index

In [ ]:
# Build FAISS IndexFlatIP (Inner Product = cosine similarity for L2-normalized vectors)
# For larger datasets (100k+), switch to IndexIVFFlat for approximate search

DIM = embeddings.shape[1]  # 384 for MiniLM
print(f'🔨 Building FAISS index (dim={DIM})...')

# IndexFlatIP: exact inner product search, perfect for <100k items
index = faiss.IndexFlatIP(DIM)

# For GPU acceleration (optional — use if Colab GPU is available)
use_gpu_faiss = device == 'cuda'
if use_gpu_faiss:
    try:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
        print('   FAISS running on GPU ✅')
    except Exception as e:
        print(f'   GPU FAISS failed ({e}), using CPU')
        use_gpu_faiss = False

# Add all embeddings to the index
index.add(embeddings)

# Convert back to CPU for saving
if use_gpu_faiss:
    index = faiss.index_gpu_to_cpu(index)

print(f'✅ FAISS index built!')
print(f'   Total vectors: {index.ntotal}')
print(f'   Index type: IndexFlatIP (exact cosine similarity)')

## 💾 Cell 8 — Save Index & Metadata

In [ ]:
# Save FAISS index to disk
INDEX_PATH = 'recommendation.index'
META_PATH = 'metadata.pkl'

faiss.write_index(index, INDEX_PATH)
print(f'✅ FAISS index saved: {INDEX_PATH} ({os.path.getsize(INDEX_PATH)/1024:.1f} KB)')

# Save metadata + model info as pickle
# We save the model name so it can be reloaded correctly
metadata_payload = {
    'records': data.to_dict('records'),   # list of all item dicts
    'model_name': MODEL_NAME,              # which embedding model was used
    'dim': DIM,                            # embedding dimension
    'total': len(data),                    # total record count
    'created_at': pd.Timestamp.now().isoformat()
}

with open(META_PATH, 'wb') as f:
    pickle.dump(metadata_payload, f)

print(f'✅ Metadata saved:   {META_PATH} ({os.path.getsize(META_PATH)/1024:.1f} KB)')
print(f'   Records: {metadata_payload["total"]}')
print(f'   Model: {MODEL_NAME}')

## 🔍 Cell 9 — Recommendation Function

In [ ]:
def recommend(query: str, top_k: int = 5, category: str = 'All', verbose: bool = True):
    """
    Get AI-powered recommendations for a natural language query.
    
    Args:
        query    : Natural language search query
        top_k    : Number of results to return
        category : Filter by category ('All','Movies','Games','Courses','Books','Software')
        verbose  : Print formatted results
    Returns:
        List of recommendation dicts with match scores
    """
    # Step 1: Encode query with same model
    q_emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    
    # Step 2: Search FAISS index
    k = min(top_k * 4, index.ntotal)  # over-fetch for category filtering
    scores, indices = index.search(q_emb, k)
    
    # Step 3: Build results with filtering
    results = []
    records = metadata_payload['records']
    
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue
        item = records[idx]
        if category != 'All' and item['category'].lower() != category.lower():
            continue
        match_pct = round(float(score) * 100, 1)
        results.append({**item, 'match_score': max(0, match_pct)})
        if len(results) >= top_k:
            break
    
    # Step 4: Pretty print results
    if verbose:
        cat_str = f'[{category}]' if category != 'All' else '[All Categories]'
        print(f'\n🔍 Query: "{query}" {cat_str}')
        print('─' * 65)
        for i, r in enumerate(results, 1):
            bar = '█' * int(r['match_score']/5) + '░' * (20 - int(r['match_score']/5))
            print(f'{i}. [{r["category"]:8}] {r["title"]}')
            print(f'   Match: {bar} {r["match_score"]}%')
            print(f'   {r["description"][:80]}...')
            print(f'   🔗 {r["url"]}')
            print()
    
    return results

print('✅ Recommendation function defined!')

## 🧪 Cell 10 — Test Recommendation Queries

In [ ]:
# ── TEST 1: AI & Machine Learning ─────────────────────────────
_ = recommend('I want to learn Artificial Intelligence', top_k=6)

In [ ]:
# ── TEST 2: Open World Fantasy Games ──────────────────────────
_ = recommend('Suggest open world fantasy RPG games', top_k=5, category='Games')

In [ ]:
# ── TEST 3: Sci-fi Movies ──────────────────────────────────────
_ = recommend('Recommend sci-fi movies about space and time travel', top_k=5, category='Movies')

In [ ]:
# ── TEST 4: Cybersecurity ──────────────────────────────────────
_ = recommend('cybersecurity hacking and network security', top_k=6)

In [ ]:
# ── TEST 5: Cross-category — Python Programming ───────────────
_ = recommend('Learn Python programming from scratch', top_k=6)

## 📦 Cell 11 — Export Files for Deployment

In [ ]:
import shutil, zipfile

# Create deployment zip
files_to_export = [
    'recommendation.index',
    'metadata.pkl',
    'datasets/movies.csv',
    'datasets/games.csv',
    'datasets/courses.csv',
    'datasets/books.csv',
    'datasets/software.csv',
]

with zipfile.ZipFile('neuralFind_index_export.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in files_to_export:
        if os.path.exists(f):
            zf.write(f)
            print(f'   ✅ Added: {f}')

print(f'\n📦 Export complete: neuralFind_index_export.zip')
print(f'   Size: {os.path.getsize("neuralFind_index_export.zip")/1024:.1f} KB')
print('\n⬇️  Download from Colab: Files panel → neuralFind_index_export.zip')

# For automatic Google Colab download:
try:
    from google.colab import files
    files.download('neuralFind_index_export.zip')
    print('\n⬇️  Auto-download triggered!')
except ImportError:
    print('\n⚠️  Not in Colab — download manually from files panel')

## 🚀 Cell 12 — Flask Integration Notes

After downloading `neuralFind_index_export.zip`:

1. Extract contents to your Flask project root
2. Verify `recommendation.index` and `metadata.pkl` are in root
3. Install requirements: `pip install -r requirements.txt`
4. Run: `python app.py`
5. Open: `http://localhost:5000`

**Note:** When using the HuggingFace model in Flask (online deployment), update `app.py` to use `SentenceTransformer` instead of TF-IDF. The `metadata.pkl` from this notebook includes `model_name` for easy reloading.

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def encode_query(query):
    return model.encode([query], normalize_embeddings=True).astype(np.float32)
```